# 01 · Análise exploratória da base Olist

**Objetivo:** entender as avaliações antes de treinar qualquer modelo.

Perguntas que este notebook responde:
1. Quantas avaliações têm texto e como as notas se distribuem?
2. Quão desbalanceado é o sentimento? (define a métrica principal)
3. Qual o tamanho dos textos? (define o `max_length` do BERTimbau)
4. O atraso na entrega está ligado a notas baixas? (valida a categoria "atraso")
5. Há textos repetidos? (justifica a divisão agrupada)

**Pré-requisito:** `python -m reviews.download` e `python -m reviews.prepare`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

from reviews.data import load_reviews
from reviews.paths import PROCESSED_REVIEWS_FILE, PROJECT_ROOT, RAW_DIR
from reviews.splits import SPLIT_NAMES

if not PROCESSED_REVIEWS_FILE.is_file():
    raise FileNotFoundError("Rode antes: python -m reviews.prepare")

df = pd.read_parquet(PROCESSED_REVIEWS_FILE)
todas = load_reviews(RAW_DIR)

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Estilo dos gráficos: marcas finas, grade discreta, uma cor por papel.
AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#8a8984"
CORES_SENTIMENTO = {"negativa": LARANJA, "neutra": CINZA, "positiva": AZUL}
plt.rcParams.update(
    {
        "figure.figsize": (8, 4),
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "grid.color": "#e5e4e0",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 10,
    }
)
pd.set_option("display.max_colwidth", 120)
df.head()

## 1. Quantas avaliações têm texto?

In [ ]:
total = todas["review_id"].nunique()
com_texto = len(df)
print(f"Avaliações (review_id únicos): {total:,}".replace(",", "."))
print(f"Com comentário:                {com_texto:,} ({com_texto / total:.1%})".replace(",", "."))

## 2. Distribuição das notas

In [ ]:
notas = pd.DataFrame(
    {
        "todas": todas.drop_duplicates("review_id")["review_score"].value_counts(normalize=True),
        "com comentário": df["review_score"].value_counts(normalize=True),
    }
).sort_index()

ax = notas.plot.bar(color=[CINZA, AZUL], width=0.75, edgecolor="white", linewidth=2, rot=0)
ax.set_title("Proporção de cada nota")
ax.set_xlabel("Nota")
ax.set_ylabel("Proporção")
ax.grid(False, axis="x")
ax.yaxis.set_major_formatter(PercentFormatter(1))
plt.tight_layout()
plt.savefig(FIG_DIR / "distribuicao_notas.png")
plt.show()
notas.style.format("{:.1%}")

Quem escreve comentário tende a dar nota mais baixa: insatisfeito fala mais.
Isso é bom para o projeto, porque as reclamações são justamente o que queremos analisar.

## 3. Desbalanceamento do sentimento

In [ ]:
sent = df["sentimento"].value_counts()
sent_pct = df["sentimento"].value_counts(normalize=True)
pd.DataFrame({"avaliações": sent, "proporção": sent_pct.map("{:.1%}".format)})

**Decisão:** como a classe positiva domina, um modelo que chuta sempre "positiva"
teria acurácia alta e seria inútil. A métrica principal do projeto é o **F1 macro**, que
dá o mesmo peso às duas classes. A nota 3 (neutra) fica fora da tarefa binária.

## 4. Tamanho dos textos

In [ ]:
df["n_palavras"] = df["texto"].str.split().str.len()
print(df.groupby("sentimento")["n_palavras"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))

fig, ax = plt.subplots()
limite = int(df["n_palavras"].quantile(0.99))
for s in ["positiva", "negativa"]:
    ax.hist(
        df.loc[df["sentimento"] == s, "n_palavras"].clip(upper=limite),
        bins=range(0, limite + 2),
        histtype="step",
        linewidth=2,
        color=CORES_SENTIMENTO[s],
        label=s,
        density=True,
    )
ax.set_title("Palavras por avaliação (cortado no percentil 99)")
ax.set_xlabel("Palavras")
ax.set_ylabel("Densidade")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "tamanho_textos.png")
plt.show()

**Para a Fase 4:** em português, o BERTimbau gera em média 1,3 a 1,5 token por palavra.
Se o percentil 99 estiver em torno de 60 palavras, `max_length=128` cobre quase tudo
sem desperdiçar memória da GPU.

## 5. Atraso na entrega e nota

In [ ]:
entregues = df[df["entregue_com_atraso"].notna()].copy()
entregues["situacao"] = entregues["entregue_com_atraso"].map(
    {True: "com atraso", False: "no prazo"}
)
resumo = entregues.groupby("situacao").agg(
    avaliacoes=("review_id", "size"),
    nota_media=("review_score", "mean"),
    pct_negativas=("sentimento", lambda s: (s == "negativa").mean()),
)
resumo.style.format({"nota_media": "{:.2f}", "pct_negativas": "{:.1%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
valores = resumo["pct_negativas"].reindex(["no prazo", "com atraso"])
barras = ax.bar(valores.index, valores.values, color=[AZUL, LARANJA], width=0.5)
ax.bar_label(barras, labels=[f"{v:.0%}" for v in valores.values], padding=3)
ax.set_title("Proporção de avaliações negativas")
ax.grid(False, axis="x")
ax.yaxis.set_major_formatter(PercentFormatter(1))
plt.tight_layout()
plt.savefig(FIG_DIR / "atraso_vs_negativas.png")
plt.show()

O atraso aparece nos dados de entrega, então a categoria **atraso** da Fase 2 pode ser
conferida contra um sinal objetivo, e não só pelo texto.

## 6. Evolução no tempo

In [ ]:
mensal = (
    df.set_index("review_creation_date")
    .resample("MS")
    .agg({"review_id": "size", "sentimento": lambda s: (s == "negativa").mean()})
    .rename(columns={"review_id": "avaliacoes", "sentimento": "pct_negativas"})
)
mensal = mensal[mensal["avaliacoes"] >= 100]  # meses com pouca amostra distorcem a taxa

fig, ax = plt.subplots()
ax.plot(mensal.index, mensal["pct_negativas"], color=LARANJA, linewidth=2, marker="o", markersize=4)
ax.set_title("Proporção de avaliações negativas por mês (meses com 100+ avaliações)")
ax.set_ylim(0, mensal["pct_negativas"].max() * 1.15)
ax.yaxis.set_major_formatter(PercentFormatter(1))
plt.tight_layout()
plt.savefig(FIG_DIR / "negativas_por_mes.png")
plt.show()

## 7. Textos repetidos

In [ ]:
repetidos = df["texto_grupo"].value_counts()
print(f"Textos distintos: {repetidos.size:,}".replace(",", "."))
pct_repetidos = (df["texto_grupo"].map(repetidos) > 1).mean()
print(f"Avaliações com texto que aparece 2+ vezes: {pct_repetidos:.1%}")
repetidos.head(15).rename("ocorrências").to_frame()

Textos curtos e genéricos se repetem muito. Se "recomendo" aparecesse no treino e no teste,
o modelo acertaria por memória. Por isso a divisão agrupa textos iguais no mesmo conjunto.

## 8. Divisão treino / validação / teste

In [ ]:
tabela = pd.crosstab(df["split"], df["sentimento"], normalize="index").reindex(list(SPLIT_NAMES))
tabela.insert(0, "avaliações", df["split"].value_counts().reindex(list(SPLIT_NAMES)))
tabela.style.format({c: "{:.1%}" for c in tabela.columns if c != "avaliações"})

As proporções de cada sentimento devem ser parecidas nos três conjuntos.

## 9. Amostra de avaliações negativas

Leia estas avaliações antes de fechar as categorias da Fase 2. A amostra vem só do
**treino**, para que o teste continue sem ser visto.

In [ ]:
negativas_treino = df[(df["sentimento"] == "negativa") & (df["split"] == "treino")]
negativas_treino.sample(25, random_state=42)[["review_score", "texto", "entregue_com_atraso"]]

## Conclusões

Preencha depois de rodar com os dados reais:

- **Avaliações com texto:** 40.641 de 98,410 (41.3%).
- **Sentimento:** 64.8% positivas, 26.5% negativas → métrica principal: F1 macro.
- **Tamanho:** percentil 99 = 60 palavras → `max_length` = 128.
- **Atraso:** 73.0% de negativas com atraso contra 18.4% no prazo.
- **Repetições:** 17.1% das avaliações têm texto repetido → divisão agrupada justificada.
- **Categorias candidatas para a Fase 2:** atraso na entrega, produto não recebido, pedido incompleto, produto errado ou diferente do anúncio, defeito ou qualidade ruim, atendimento / cancelamento / reembolso, outro. A confirmar lendo a amostra da seção 9.